# Knowledge-source assistant

Notebook ini memilih satu sumber untuk setiap pertanyaan: Wikipedia untuk pengetahuan umum, Arxiv untuk paper atau topik ilmiah, dan retriever dokumentasi LangSmith untuk pertanyaan khusus LangSmith. Jawaban akhir hanya dibuat dari dokumen hasil pencarian dan menyertakan penanda sumber.

Jalankan notebook dari direktori utama proyek setelah memasang dependensi dari `requirements.txt` dan mengatur `GOOGLE_API_KEY` di `.env`. Notebook membutuhkan koneksi internet untuk mengambil dokumentasi dan hasil pencarian.

In [11]:
import os
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_community.retrievers import ArxivRetriever, WikipediaRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise RuntimeError("GOOGLE_API_KEY belum diatur. Tambahkan ke file .env.")

os.environ["GOOGLE_API_KEY"] = google_api_key
os.environ.setdefault("USER_AGENT", "knowledge-source-assistant/1.0")

import nest_asyncio
nest_asyncio.apply()

In [12]:
class RouteDecision(BaseModel):
    source: Literal["wikipedia", "arxiv", "langsmith_docs"] = Field(
        description="Satu sumber paling sesuai untuk mencari bukti jawaban."
    )
    reason: str = Field(description="Alasan singkat pemilihan sumber.")


model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=google_api_key,
    temperature=0,
)
router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Pilih tepat satu sumber. Gunakan langsmith_docs hanya untuk pertanyaan tentang LangSmith, fitur, dokumentasi, atau API-nya. "
            "Gunakan arxiv untuk paper penelitian atau pertanyaan yang membutuhkan literatur ilmiah. "
            "Gunakan wikipedia untuk pengetahuan umum. Jangan menjawab pertanyaan; tugasmu hanya memilih sumber.",
        ),
        ("human", "Pertanyaan: {question}"),
    ]
)
router = router_prompt | model.with_structured_output(RouteDecision)

In [13]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://docs.langchain.com/langsmith/"
)

docs = loader.load()

print(len(docs))
print(len(docs[0].page_content))
print(docs[0].page_content[:500])


1
2706






LangSmith Observability - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferen


In [14]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_community.retrievers import ArxivRetriever

In [15]:
wikipedia_retriever = WikipediaRetriever(
    top_k_results=3,
    lang="en"
)

arxiv_retriever = ArxivRetriever(
    top_k_results=3
)

In [16]:
print("wikipedia_retriever" in globals())
print("arxiv_retriever" in globals())
print("langsmith_retriever" in globals())

True
True
True


## Siapkan tiga sumber

Wikipedia dan Arxiv dicari saat ada pertanyaan. Dokumentasi LangSmith dimuat dari beberapa halaman Markdown resmi tentang observability agar pemuatan tetap ringan, lalu diindeks untuk pencarian relevansi.

In [17]:
from langchain_community.document_loaders import WebBaseLoader

langsmith_doc_urls = [
    "https://docs.langchain.com/langsmith/observability.md",
    "https://docs.langchain.com/langsmith/observability-concepts.md",
    "https://docs.langchain.com/langsmith/observability-quickstart.md",
    "https://docs.langchain.com/langsmith/observability-llm-tutorial.md",
    "https://docs.langchain.com/langsmith/view-traces.md",
    "https://docs.langchain.com/langsmith/filter-traces.md",
]
langsmith_loader = WebBaseLoader(
    web_paths=langsmith_doc_urls,
    requests_per_second=2,
    requests_kwargs={"timeout": 30},
    raise_for_status=True,
    show_progress=False,
)
langsmith_pages = [
    document
    for document in langsmith_loader.load()
    if document.page_content.strip()
]

if not langsmith_pages:
    raise RuntimeError(
        "Tidak ada halaman dokumentasi LangSmith yang berhasil dimuat."
    )

print(f"Jumlah halaman berhasil dimuat: {len(langsmith_pages)}")

# Split documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)

langsmith_chunks = splitter.split_documents(langsmith_pages)

if not langsmith_chunks:
    raise RuntimeError(
        "Dokumentasi LangSmith tidak menghasilkan chunk untuk diindeks."
    )

print(f"Jumlah chunk: {len(langsmith_chunks)}")

# Embedding
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=google_api_key,
)

# Vector Store
langsmith_vector_store = FAISS.from_documents(
    langsmith_chunks,
    embeddings,
)

langsmith_retriever = langsmith_vector_store.as_retriever(
    search_kwargs={"k": 4}
)

# Retrievers
retrieval_tools = {
    "wikipedia": wikipedia_retriever,
    "arxiv": arxiv_retriever,
    "langsmith_docs": langsmith_retriever,
}

print("Semua retriever berhasil dibuat.")

Jumlah halaman berhasil dimuat: 6
Jumlah chunk: 68
Semua retriever berhasil dibuat.


In [18]:
def format_evidence(documents):
    evidence = []
    sources = []

    for index, document in enumerate(documents, start=1):
        metadata = document.metadata
        title = metadata.get("title") or metadata.get("Title") or "Sumber tanpa judul"
        url = (
            metadata.get("source")
            or metadata.get("entry_id")
            or metadata.get("Entry ID")
            or metadata.get("url")
        )
        if url and not str(url).startswith(("http://", "https://")):
            url = f"https://arxiv.org/abs/{url}"

        marker = f"[S{index}]"
        sources.append({"marker": marker, "title": title, "url": url})
        evidence.append(
            f"{marker} {title}\nURL: {url or 'URL tidak tersedia'}\n{document.page_content}"
        )

    return "\n\n".join(evidence), sources

In [19]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Jawab dalam bahasa yang digunakan penanya. Jawab hanya berdasarkan bukti pada konteks. "
            "Cantumkan penanda [S1], [S2], dan seterusnya pada klaim yang didukung. "
            "Jika konteks tidak cukup untuk menjawab, katakan dengan jelas bahwa informasi tidak ditemukan; jangan menebak atau memakai pengetahuan di luar konteks.",
        ),
        ("human", "Konteks hasil pencarian:\n{context}\n\nPertanyaan: {question}"),
    ]
)
answer_chain = answer_prompt | model


def ask_assistant(question: str) -> dict:
    if not question.strip():
        raise ValueError("Pertanyaan tidak boleh kosong.")

    decision = router.invoke({"question": question})
    documents = retrieval_tools[decision.source].invoke(question)
    context, sources = format_evidence(documents)

    if not context:
        answer = "Informasi yang relevan tidak ditemukan pada sumber yang dipilih."
    else:
        answer = answer_chain.invoke(
            {"question": question, "context": context}
        ).content

    return {
        "source": decision.source,
        "reason": decision.reason,
        "answer": answer,
        "references": sources,
    }

## Coba assistant

Ubah pertanyaan berikut atau panggil `ask_assistant("...")` untuk pertanyaan lain. Hasil menampilkan sumber yang dipilih, jawaban, dan referensinya.

In [20]:
question = "Apa itu LangSmith dan bagaimana tracing membantu proses debugging?"
result = ask_assistant(question)

print(f"Sumber dipilih: {result['source']} ({result['reason']})")
print(f"\nJawaban:\n{result['answer']}")
print("\nReferensi:")
for reference in result["references"]:
    print(f"{reference['marker']} {reference['title']} - {reference['url'] or 'URL tidak tersedia'}")

Sumber dipilih: langsmith_docs (Pertanyaan berkaitan langsung dengan LangSmith dan fitur tracing-nya untuk debugging.)

Jawaban:
Berdasarkan konteks yang diberikan:

### Apa itu LangSmith?
LangSmith (melalui fitur *LangSmith Observability*) adalah platform yang memberikan visibilitas penuh ke dalam aplikasi LLM, mulai dari pelacakan (*traces*) individual hingga metrik performa di tingkat produksi [S3]. Platform ini digunakan untuk melakukan instrumentasi aplikasi LLM, menginvestigasi *traces*, serta memantau performa [S3]. Selain observabilitas, LangSmith juga mencakup evaluasi, *prompt engineering*, dan *deployment* [S2].

---

### Bagaimana Tracing Membantu Proses Debugging?
*Tracing* menyimpan catatan riwayat mengenai apa saja yang dilakukan oleh *agent* Anda di lingkungan produksi yang dapat digunakan secara langsung untuk mendebug kegagalan [S3]. Proses debugging terbantu melalui cara-cara berikut:

1. **Pemeriksaan Detail Komponen (*Details View*):** Tampilan detail berfungsi seb